# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. The dataset is defined by a Croissant schema and demonstrates AI-ready data standards for data infrastructure in Africa.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
* https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.
Below, we print out the `@id` for each record set, their fields and columns.

In [ ]:
# Display all record sets and their structure
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, Name: {rs.name if hasattr(rs,'name') else 'N/A'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    * Field @id: {field.id}, Name: {field.name}, DataType: {getattr(field, 'data_type', 'N/A')}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    * Column @id: {col.id}, Name: {col.name}, Source: {col.source if hasattr(col, 'source') else 'N/A'}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All entities are referenced by their `@id`. Below, we load all record sets found above.

In [ ]:
# Collect record set IDs
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for RecordSet {rs_id}")

# Display columns of the main record set (choose the first one as example)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns in main RecordSet ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Common data processing steps: filtering records, normalizing numeric fields, and grouping by key attributes.

Below, we'll select a numeric field (by its `@id`), filter by a threshold, normalize it, and group by a categorical field.

In [ ]:
# Example EDA operations
# First, list available numeric fields for the main recordset
main_df = dataframes[main_record_set_id]

# Find a numeric field
numeric_fields = [col for col in main_df.columns if main_df[col].dtype in ['int64','float64']]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # fallback example
    numeric_field_id = 'gad7_score' if 'gad7_score' in main_df.columns else main_df.columns[0]

print(f"Using numeric field: {numeric_field_id}")

threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
if not filtered_df.empty:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a categorical field to group by
group_fields = [col for col in main_df.columns if main_df[col].dtype == 'object']
group_field_id = 'gender' if 'gender' in main_df.columns else (group_fields[0] if group_fields else None)

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we visualize the distribution of the numeric field and its breakdown by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Plot boxplot grouped by group_field_id if available
if group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. We reviewed the data structure via record set and field `@id`s, loaded values into DataFrames, applied basic filtering and normalization, and visualized numeric data distributions and grouping by demographic attributes. These steps demonstrate reproducible, AI-ready practices using Croissant schema-based datasets for public health research.